In [ ]:
from pathlib import Path
import json
import warnings
import sys
from typing import List

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "main" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "finetuning_results.csv"

df_all = pd.read_csv(results_file)

In [ ]:
df_all.columns

In [ ]:
benchmark_order = {
    k: i for i, k in enumerate(BENCHMARK_NAME_MAPPING.keys())
}

In [ ]:
def get_base_model_id(model_id: str) -> str:
    arch_map = {
        'deit_small': 'deit_small_imagenet_full_seed-0',
        'convnext_small': 'convnext_small_imagenet_full_seed-0',
        'resnet50': 'resnet50_imagenet_full',
    }
    return arch_map[model_id.split('-')[0]]

In [ ]:
GROUPBY_COLS = ['model_id', 'benchmark_name', 'seed', 'finetuning_dataset']

def filter_df(df_orig: pd.DataFrame, exp_type:str, groupby_cols: List[str]=GROUPBY_COLS, data_pct:int=None, model_id:str=None, set_base_model_id:bool=True) -> pd.DataFrame:
    df = df_orig.copy()
    df = df[df['exp_type'] == exp_type]
    if data_pct is not None:
        df = df[df['data_pct'] == data_pct]
    if model_id is not None:
        df = df[df['model_id'] == model_id]
        
    df = apply_hiearchical_aggregation(
        df = df,
        groupby_cols = groupby_cols,
        agg_cols = METRICS_COMPACT + ['task_evaluation_acc'],
        agg_func = 'mean'
    )
    
    if set_base_model_id:
        df['base_model_id'] = df['model_id'].apply(get_base_model_id)
    
    return df

In [ ]:
LEFT_ON_COLS = ['base_model_id', 'benchmark_name']
RIGHT_ON_COLS = ['model_id', 'benchmark_name']
SUFFIXES = ('_finetuned', '_baseline')
SORT_BY_COLS = ['finetuning_dataset', 'benchmark_name']

def get_df_diffs(df1: pd.DataFrame, df2: pd.DataFrame, left_on_cols: List[str]=LEFT_ON_COLS, right_on_cols: List[str]=RIGHT_ON_COLS, suffixes: tuple=SUFFIXES, sort_by_cols: List[str]=SORT_BY_COLS) -> pd.DataFrame:
    df_merged = df1.merge(
        right=df2, 
        left_on=left_on_cols, 
        right_on=right_on_cols, 
        suffixes=suffixes
    )
    
    for metric in METRICS_COMPACT + ['task_evaluation_acc']:
        df_merged[f'{metric}_diff'] = df_merged[f'{metric}_finetuned'] - df_merged[f'{metric}_baseline']
    
    sorting_cols = []
    if 'finetuning_dataset' in sort_by_cols:
        sorting_cols.append('finetuning_dataset_order')
        df_merged['finetuning_dataset_order'] = df_merged['finetuning_dataset'].map(benchmark_order)
    if 'benchmark_name' in sort_by_cols:
        sorting_cols.append('benchmark_order')
        df_merged['benchmark_order'] = df_merged['benchmark_name'].map(benchmark_order)

    if sorting_cols:
        df_merged = df_merged.sort_values(
            by=sorting_cols
        )
        
    return df_merged

# Architecture Comparison

In [ ]:
df_finetuned_arch = df_all[
    (df_all["exp_type"] == 'finetuning_data_pct') & (df_all["data_pct"] == 100)
    | (df_all["exp_type"] == 'finetuning_architecture')
    ].copy()

df_finetuned_arch = df_finetuned_arch[df_finetuned_arch["data_pct"] == 100]
df_finetuned_arch = apply_hiearchical_aggregation(
    df = df_finetuned_arch,
    groupby_cols = ['model_id', 'seed', 'finetuning_dataset'],
    agg_cols = METRICS_COMPACT + ['task_evaluation_acc'],
    agg_func = 'mean'
)

df_finetuned_arch['base_model_id'] = df_finetuned_arch['model_id'].apply(get_base_model_id)

df_finetuned_arch.shape

In [ ]:
df_baseline_arch = filter_df(
    df_orig=df_all,
    exp_type='finetuning_baseline',
    groupby_cols=['model_id'],
    set_base_model_id=False
)
df_baseline_arch

In [ ]:
df_diff_arch = df_finetuned_arch.merge(
    df_baseline_arch,
    left_on=['base_model_id'],
    right_on=['model_id'],
    suffixes=('_finetuned', '_baseline')
)

for metric in METRICS_COMPACT:
    df_diff_arch[f'{metric}_diff'] = df_diff_arch[f'{metric}_finetuned'] - df_diff_arch[f'{metric}_baseline']

## Add average improvement per finetuning dataset
df_diff_arch_avg = df_diff_arch.groupby(['model_id_baseline', 'base_model_id', 'seed']).agg({
    f'{metric}_diff': 'mean' for metric in METRICS_COMPACT
}).reset_index()
df_diff_arch_avg['finetuning_dataset'] = 'benchmark_average'
df_diff_arch = pd.concat([df_diff_arch, df_diff_arch_avg], ignore_index=True)


df_diff_arch['finetuning_dataset_order'] = df_diff_arch['finetuning_dataset'].map(benchmark_order)
# df_diff_arch['benchmark_order'] = df_diff_arch['benchmark_name'].map(benchmark_order)

df_diff_arch['model_name'] = df_diff_arch.model_id_baseline.map({
    'deit_small_imagenet_full_seed-0': 'ViT-S',
    'convnext_small_imagenet_full_seed-0': 'ConvNeXt-S',
    'resnet50_imagenet_full': 'ResNet-50',
})

df_diff_arch = df_diff_arch.sort_values(
    by=['finetuning_dataset_order', 'model_name']
)

df_diff_arch.finetuning_dataset = df_diff_arch.finetuning_dataset.map(BENCHMARK_NAME_MAPPING)


df_diff_arch.shape

In [ ]:
zoom = 0.75
zoom = 0.6
figsize = (15,  10)
zoom = 1.0
zoom = 0.6
figsize = (6,  6)
figsize = (8,  6)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)
ax = axes

arch_families = df_diff_arch['model_name'].unique()
n_arch = len(arch_families)

palette = {
    'ViT-S': ARCHITECUTURE_FAMILY_COLORS['ViT'],
    'ConvNeXt-S': ARCHITECUTURE_FAMILY_COLORS['ConvNeXt'],
    'ResNet-50': ARCHITECUTURE_FAMILY_COLORS['ResNet'],
}


data = df_diff_arch[df_diff_arch.finetuning_dataset == 'Average'].copy()
sns.barplot(
    x='finetuning_dataset',
    y='pearsonr_nc_diff',
    hue='model_name',
    palette=palette,
    # data=df_diff_arch[df_diff_arch.model_id_finetuned.notna()],
    data=df_diff_arch,
    ax=axes
)

    
# ax.set_title(f'Finetuned on: {benchmark_names[ds]}', fontsize=16, fontweight='bold')
ax.set_title(f'Architecture Comparison', fontsize=20, fontweight='bold')
# ax.set_ylabel(f"Change in Average Alignment", fontsize=16, fontweight='bold')
ax.set_ylabel(r"$\Delta$ Average Alignment", fontsize=16, fontweight='bold')
ax.set_xlabel(f"Finetuning Dataset", fontsize=16, fontweight='bold')


# Rotate x labels for better readability
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

ax.set_ylim(-0.005, 0.015)
ax.grid(axis='y', linestyle='--', alpha=0.7, zorder=-100)


# remove legend panel background
ax.legend(
    # title='Model', 
    title=None, 
    loc='upper center', 
    # frameon=False
)


ax.spines[['top', 'right']].set_visible(False)



figures_dir = save_dir
fig_name = 'fig4-finetuning-improvements-arch_comparison'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)

# Regions

In [ ]:
df_regions = df_all[
    (df_all["exp_type"] == 'finetuning_region') | 
    ((df_all["exp_type"] == 'finetuning_data_pct') & (df_all["data_pct"] == 100) & (df_all["finetuning_dataset"].isin(['tvsd', 'things_fmri', 'nsd_func1pt8mm_individualROIs'])))
].copy()
# df_finetuned = df_finetuned[df_finetuned["data_pct"] == 100]

df_regions = apply_hiearchical_aggregation(
    df = df_regions,
    groupby_cols = ['model_id', 'seed', 'finetuning_regions', 'finetuning_dataset'],
    agg_cols = METRICS_COMPACT + ['task_evaluation_acc'],
    agg_func = 'mean'
)

df_regions['base_model_id'] = df_regions['model_id'].apply(get_base_model_id)

df_regions.shape


In [ ]:
df_baseline_regions = filter_df(
    df_orig=df_all,
    exp_type='finetuning_baseline',
    model_id='deit_small_imagenet_full_seed-0',
    groupby_cols=['model_id'],
    set_base_model_id=False
)
df_baseline_regions.shape

In [ ]:
df_diff_region = get_df_diffs(
    df1=df_regions,
    df2=df_baseline_regions,
    left_on_cols=['base_model_id', ],
    right_on_cols=['model_id',],
    suffixes=('_finetuned', '_baseline'),
    sort_by_cols=['finetuning_dataset']
)
df_diff_region.shape

In [ ]:
regions_names = {
    'V1': 'V1',
    'V4': 'V4',
    'IT': 'IT',
    'V1_V4_IT': 'V1+V4+IT',
    'early': 'Early',
    'midventral': 'MidVentral',
    'ventral': 'Ventral',
    'early_midventral_ventral': 'Early+Mid+Late',
}
# regions_names = {
#     'V1': 'V1',
#     'V4': 'V4',
#     'IT': 'IT',
#     'V1_V4_IT': 'All regions',
#     'early': 'Early',
#     'midventral': 'MidVentral',
#     'ventral': 'Ventral',
#     'early_midventral_ventral': 'All regions',
# }

regions_order = {
    k: i for i, k in enumerate(regions_names.keys())
}

In [ ]:

base_colors = {
    "T-fMRI":   BENCHMARK_COLORS['things_fmri'],
    "NSD-fMRI": BENCHMARK_COLORS['nsd_func1pt8mm_individualROIs'],
    "TVSD-EP":  BENCHMARK_COLORS['tvsd'],
}

def blend_with_white(hex_color, t):
    """t in [0,1]: 0 -> original, 1 -> white"""
    rgb = np.array(mcolors.to_rgb(hex_color))
    return mcolors.to_hex((1 - t) * rgb + t * np.ones(3))

def per_dataset_roi_palette(data, dataset_order, hue_order, base_colors,
                            light=0.25, dark=0.00):
    """
    hue values look like '{ROI}-{DATASET}' (e.g., 'V1-T-fMRI').
    Build a separate light->dark gradient *within each dataset* over its ROIs only.
    Keep light small so colors remain close to the dataset base.
    """
    palette = {}

    for ds in dataset_order:
        base = base_colors.get(ds, "#777777")

        # hues that belong to this dataset, in the SAME order as your hue_order
        hues_ds = [h for h in hue_order if h.endswith(f"-{ds}")]

        # unique ROI list for this dataset, preserving order
        rois_ds = []
        for h in hues_ds:
            roi = h.rsplit("-", 1)[0]  # split off dataset suffix safely
            if roi not in rois_ds:
                rois_ds.append(roi)

        n = len(rois_ds)
        if n == 0:
            continue

        ts = np.linspace(light, dark, n)  # small range => close to base color
        roi2col = {roi: blend_with_white(base, t) for roi, t in zip(rois_ds, ts)}

        for h in hues_ds:
            roi = h.rsplit("-", 1)[0]
            palette[h] = roi2col[roi]

    return palette


In [ ]:
zoom = 1
figsize = (8,  6)
zoom = 0.6
figsize = (6,  8)
figsize = (8,  8)
figsize = (figsize[0] * zoom, figsize[1] * zoom)
fig, ax = plt.subplots(
    1, 1, 
    figsize=figsize, 
    dpi=300
)

data = df_diff_region.copy()
data["finetuning_dataset"] = data["finetuning_dataset"].map(BENCHMARK_NAME_MAPPING)
data["finetuning_regions_order"] = data["finetuning_regions"].map(regions_order)
data["finetuning_regions"] = data["finetuning_regions"].map(regions_names)
data["finetuning_regions"] = data.apply(lambda row: f'{row["finetuning_regions"]}-{row["finetuning_dataset"]}', axis=1)
data = data.sort_values(
    by=["finetuning_dataset_order", "finetuning_regions_order"]
)

dataset_order = data["finetuning_dataset"].unique()
region_order = data["finetuning_regions"].unique()


palette = per_dataset_roi_palette(
    data=data,
    dataset_order=dataset_order,   # your x order
    hue_order=region_order,        # your hue_order (finetuning_regions)
    base_colors=base_colors,
    light=0.22,  # try 0.15–0.30; lower = closer to base
    dark=0.00,
)

sns.barplot(
    data=data,
    x="finetuning_dataset",
    y="pearsonr_nc_diff",
    hue="finetuning_regions",
    order=dataset_order,
    hue_order=region_order,
    ax=ax,
    width=10,
    palette=palette
)

# ---- remove default legend & x ticks ----
ax.legend_.remove()
ax.set_xticks([])

# ---- y positions for text ----
ymin, ymax = ax.get_ylim()
region_y = ymin - 0.01 * (ymax - ymin)
dataset_y = ymin - 0.22 * (ymax - ymin)
dataset_y = ymin - 0.27 * (ymax - ymin)

# ---- region labels under each bar (45°) ----
for container, region in zip(ax.containers, region_order):
    for bar in container:
        x = bar.get_x() + bar.get_width() / 2 + 0.3
        ax.text(
            x,
            region_y,
            region.split('-')[0],  # remove dataset from label
            ha="right",
            va="top",
            rotation=45,
            fontsize=9,
        )


x_shifts = [
    -3.4,
    0.1,
    # 4.5
    3.5
]
# ---- dataset labels centered under each group ----
for i, dataset in enumerate(dataset_order):
    # bar groups are centered on integer x positions: 0, 1, 2, ...
    ax.text(
        i + x_shifts[i],
        dataset_y,
        dataset,
        ha="center",
        va="top",
        fontsize=14,
        fontweight="bold",
    )

# ---- expand y-limits so text is visible ----
ax.set_ylim(0, ymax)
plt.subplots_adjust(bottom=0.32)

# ---- styling ----
ax.set_title("Fine-Tuning Regions", fontsize=20, fontweight="bold")
ax.set_ylabel("Changing Average Alignment", fontsize=16, fontweight="bold")
ax.set_ylabel(r"$\Delta$ Average Alignment", fontsize=16, fontweight='bold')
ax.set_xlabel("")  # handled manually

ax.margins(x=0.1) 

ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.spines[["top", "right"]].set_visible(False)


figures_dir = save_dir
fig_name = 'fig4-finetuning-improvements-regions'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)